# 激活函数层

上一章末尾留下了一个悬念：**没有激活函数，无论堆叠多少层线性层，整体仍然等价于一个单层线性变换。**

这可以通过代数直接验证。设两层线性网络为：

$$
h = xW_1^T + b_1, \quad p = hW_2^T + b_2
$$

将 $h$ 代入 $p$：

$$
p = (xW_1^T + b_1)W_2^T + b_2 = x(W_1^TW_2^T) + (b_1W_2^T + b_2)
$$

结果仍然是一个线性变换 $p = xW' + b'$，只是用了新的权重矩阵 $W'$ 和偏置 $b'$。
无论叠多少层，都可以被折叠成一层，多余的层什么也没做。

要打破这一限制，每两层线性层之间需要插入一个**非线性函数**——即**激活函数**。

激活函数是人工神经元的核心组成部分，但在深度学习实践中，
通常被设计成**独立的层**，与线性层组合使用，方便灵活地插拔和替换。

本章实现最常用的激活函数：**ReLU**（Rectified Linear Unit，修正线性单元）。

In [1]:
from abc import ABC, abstractmethod

import numpy as np

np.random.seed(99)  # 固定随机种子，确保结果可复现

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()
        for parent in self.parents:
            parent.backward()

    @property
    def size(self):
        return self.data.shape[-1]

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.train_features = None
        self.train_labels = None
        self.test_features = None
        self.test_labels = None
        self.load()
        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self): pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    @property
    def shape(self):
        return Tensor(self.features).size, Tensor(self.labels).size

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size
        return Tensor(self.features[start:end]), Tensor(self.labels[start:end])

### 基础层

In [4]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor): pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor): pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for param in self.parameters:
            param.grad = np.zeros_like(param.data)

    @abstractmethod
    def step(self): pass

### 基础模型

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs): pass

    @abstractmethod
    def test(self, dataset): pass

## 数据

### 冰激凌销量数据集

In [8]:
class LRDataset(Dataset):

    def load(self):
        self.train_features = [[22.5, 72.0],
                               [31.4, 45.0],
                               [19.8, 85.0],
                               [27.6, 63.0]]
        self.train_labels = [[95],
                             [210],
                             [70],
                             [155]]
        self.test_features = [[28.1, 58.0]]
        self.test_labels = [[165]]

## 模型

### 线性层

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.random.rand(out_size, in_size) / in_size)
        self.bias = Tensor(np.random.rand(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.data.shape}; bias{self.bias.data.shape}]'

### 顺序层

In [10]:
class Sequential(Layer):

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x: Tensor):
        for layer in self.layers:
            x = layer(x)
        return x

    @property
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters]

    def __repr__(self):
        return '\n'.join(repr(layer) for layer in self.layers)

### ReLU 激活函数

**ReLU**（Rectified Linear Unit）是目前最常用的隐藏层激活函数，定义极为简单：

$$
\text{ReLU}(x) = \max(0, x)
$$

负数输入直接归零，正数输入原样通过。
正是这个简单的非线性截断，打破了多层线性叠加的等价性，
让深层网络获得了拟合任意复杂函数的能力。

**ReLU 的梯度**：

$$
\frac{d\,\text{ReLU}(x)}{dx} = \begin{cases} 1 & x > 0 \\ 0 & x \leq 0 \end{cases}
$$

在代码中，用 `(a.data > 0)` 来构造这个 0/1 掩码——
当激活值 `a.data > 0` 时，说明原始输入也 `> 0`，梯度正常传递；否则梯度截断为 0。

链式法则的完整形式：

$$
\frac{dL}{dx} = \underbrace{a.\text{grad}}_{\text{上游梯度}} \times \underbrace{(a.\text{data} > 0)}_{\text{局部梯度}}
$$

``💡 ReLU 的一个已知缺点是**死亡 ReLU**（Dying ReLU）：如果某个神经元的输入在训练中始终为负，它的梯度永远是 0，参数永远无法更新，神经元就"死"了。Leaky ReLU、ELU、Swish 等变体正是为了解决这个问题而设计的。``

---

加入 ReLU 后，计算图新增了**激活值张量** $a$：

```{figure} images/relu-compu-graph.png
:align: center
:width: 480px
**图例：两层网络（含 ReLU）的计算图结构**

* $x$：起点（特征值）；
* $h$：隐藏层输出（线性变换结果）；
* $a$：激活值（ReLU 输出）；
* $p$：预测值（输出层输出）；
* $L$：终点（损失值）；
* $w_1, b_1, w_2, b_2$：叶节点（模型参数，不参与 parents 链路遍历）。

```

ReLU 层自身**没有可训练参数**（`parameters` 默认返回空列表），
梯度直接穿透 ReLU 层，根据激活值的正负选择性地传递给上游。

In [11]:
class ReLU(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.maximum(0, x.data))

        def gradient_fn():
            # 上游梯度 × 局部梯度（激活值 > 0 的位置为 1，其余为 0）
            x.grad += a.grad * (a.data > 0)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### 均方误差损失函数

In [12]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            n = len(y.data)
            p.grad += -2 * (y.data - p.data) / n

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 随机梯度下降优化器

In [13]:
class SGDOptimizer(Optimizer):

    def step(self):
        for param in self.parameters:
            param.data -= param.grad * self.lr

### 神经网络模型

In [14]:
class NNModel(Model):

    def train(self, dataset, epochs):
        dataset.train()
        log_interval = max(1, epochs // 10)

        for epoch in range(epochs):
            epoch_loss = 0
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.zero_grad()  # 1. 清零梯度
                predictions = self.layer(features)  # 2. 前向传播
                loss = self.loss_fn(predictions, labels)
                loss.backward()  # 3. 自动微分
                self.optimizer.step()  # 4. 更新参数
                epoch_loss += float(loss.data)

            if epoch % log_interval == 0 or epoch == epochs - 1:
                avg = epoch_loss / len(dataset)
                print(f'epoch {epoch:4d}/{epochs}  train_loss={avg:.4f}')

    def test(self, dataset):
        dataset.eval()
        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 设置

### 学习率

In [15]:
LEARNING_RATE = 0.00001

### 批大小

In [16]:
BATCH_SIZE = 2

### 轮次

In [17]:
EPOCHS = 1000

## 训练

### 建模

在 `Sequential` 层列表中，只需把 `ReLU()` 插入两个线性层之间，
就完成了一个标准的两层全连接神经网络：

```
x_in → Linear(2→4) → ReLU → Linear(4→1) → 预测值
```

ReLU 没有参数，不影响 `layer.parameters` 的收集，
也不需要对模型类做任何修改。

In [18]:
dataset = LRDataset(BATCH_SIZE)

layer = Sequential([
    Linear(dataset.shape[0], 4),  # 隐藏层：2 → 4
    ReLU(),  # 非线性激活
    Linear(4, dataset.shape[1]),  # 输出层：4 → 1
])
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(model.layer)

Linear[weight(4, 2); bias(4,)]
ReLU[]
Linear[weight(1, 4); bias(1,)]


### 迭代训练

In [19]:
model.train(dataset, EPOCHS)

epoch    0/1000  train_loss=16861.8596
epoch  100/1000  train_loss=39.1528
epoch  200/1000  train_loss=5.4415
epoch  300/1000  train_loss=5.3466
epoch  400/1000  train_loss=4.5209
epoch  500/1000  train_loss=4.4357
epoch  600/1000  train_loss=4.3246
epoch  700/1000  train_loss=4.5530
epoch  800/1000  train_loss=4.7183
epoch  900/1000  train_loss=4.5825
epoch  999/1000  train_loss=4.4378


## 验证

### 测试

In [20]:
predictions, loss = model.test(dataset)
print(f'prediction: {predictions}')
print(f'test loss:  {loss}')

prediction: Tensor([[164.86381116]])
test loss:  Tensor(0.018547399366677222)


加入 ReLU 后，测试损失降至约 `0.018`，预测值约 `164.9`，真实值为 `165`。

对比各章结果：

| 章节 | 网络结构 | 测试损失 |
|------|---------|--------|
| 第七章 | 单层线性，1000 epoch | 2.18 |
| 第八章 | 两层线性（无激活），1000 epoch | 2.51 |
| **第九章** | **线性 + ReLU + 线性，1000 epoch** | **0.018** |

激活函数带来了约 **100 倍**的性能提升。
这清晰地展示了非线性变换的价值：即使是最简单的 ReLU，
也足以让网络学到训练数据中更细致的结构。

``💡 需要注意的是，当前训练集只有 4 个样本，测试集只有 1 个样本。
如此低的测试损失，部分原因可能是模型对训练数据的**记忆**（过拟合），
而非真正学到了温度湿度与销量之间的普遍规律。
在真实场景中，需要用更多数据和正则化手段来验证模型的泛化能力。``

---

准确地说，我们的神经网络训练框架现在已经能够很好地支持
**全连接神经网络**（Fully Connected Neural Network），
并解决**数值回归**问题。

## 结束语

至此，我们从零开始，完整构建了一个神经网络训练框架，
并用它重现了第一部分的冰激凌销量预测任务。

回顾整个框架的六个核心类：

| 类 | 职责 |
|---|---|
| `Tensor` | 数据容器，携带梯度，构成计算图 |
| `Dataset` | 数据管理，训练/测试模式切换，批次迭代 |
| `Layer` / `Sequential` | 前向传播，复合层支持多层网络 |
| `Loss` | 量化预测误差，提供梯度起点 |
| `Optimizer` | 根据梯度更新参数 |
| `Model` | 整合训练流程，对外提供简洁接口 |

通过这个过程，我们深入理解了神经网络的两个核心机制：
**计算图**（前向传播如何建立节点间的依赖关系）
和**自动微分**（反向传播如何沿 `parents` 链路自动传递梯度）。

下一部分将在这个框架的基础上，继续探索更多神经网络模型和应用场景。